# L15 · 组合构建（Portfolio Construction）

**预计时长**：90 min | **难度**：⭐⭐⭐⭐⭐ | **前置**：L14

## 本节目标
1. 从「选股」跃迁到「配置」：仓位权重如何决定
2. 三大主流方案：等权 / 反向波动加权 / Markowitz 最小方差
3. 组合再平衡（rebalance）：定期 vs 触发式
4. 评估：组合层面的夏普 / 回撤 / 多样化收益

## 第 1 段：金融概念

### 为什么权重重要
选出 5 只好股票，但全仓押 1 只 → 仍是单股风险。
权重决定了**组合的波动率、回撤、多样化效果**。

### 三大权重方案
| 方案 | 公式 | 优点 | 缺点 |
|------|------|------|------|
| **等权** | w_i = 1/N | 最简单、稳健 | 忽略各股风险差异 |
| **反向波动加权** | w_i ∝ 1/σ_i | 低波动股权重大 | 对 σ 估计敏感 |
| **Markowitz 最小方差** | argmin w'Σw s.t. Σw=1 | 理论最优 | 对协方差矩阵敏感，过拟合 |

### Markowitz 现代组合理论（1952）
- 核心：在给定收益下最小化风险，或给定风险下最大化收益
- 有效前沿（Efficient Frontier）：风险-收益平面的上沿曲线
- 局限：σ 和相关性估计误差被放大，过去 ≠ 未来

### A 股组合实战
- **5-10 只股票足够**：分散边际收益递减
- **跨行业**：避免行业暴雷全军覆没
- **定期再平衡**（季度/半年）：让组合不偏离目标权重
- **不要过度优化**：Markowitz 对噪声极度敏感

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation（共享 _data/_style）+ project root
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (
    _cwd / 'learning' / 'phase1_foundation'
    if (_cwd / 'learning' / 'phase1_foundation' / '_data.py').exists()
    else _cwd.parent / 'phase1_foundation'
)
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data
from qtrader import (
    BacktestEngine, CostModel,
    DualMAStrategy, TurtleStrategy, GridStrategy,
    print_comparison_table,
)

## 第 2 段：复用 L14 的 panel

In [ ]:
codes_names = [
    ('002594', '比亚迪'), ('600519', '贵州茅台'),
    ('002624', '完美世界'), ('300750', '宁德时代'),
    ('000001', '平安银行'),
]
codes = [c for c, _ in codes_names]
frames = {c: get_stock_data(c).set_index('date')['close'] for c in codes}
panel = pd.DataFrame(frames).sort_index().ffill()
rets = panel.pct_change().dropna()
print(f"panel shape: {panel.shape}")

## 第 3 段：三种权重方案

In [ ]:
# 方案 1: 等权
N = len(codes)
w_equal = pd.Series(1/N, index=codes)

# 方案 2: 反向波动加权
vol = rets.std() * np.sqrt(252)
w_inv_vol = (1 / vol) / (1 / vol).sum()

# 方案 3: Markowitz 最小方差（解析解，无需 scipy）
# min w'Σw s.t. Σw=1 的闭式解：w = Σ^(-1) 1 / (1' Σ^(-1) 1)
cov = rets.cov() * 252  # 年化协方差
inv_cov = np.linalg.pinv(cov.values)  # pseudo-inverse 防奇异
ones = np.ones(N)
w_mkv_values = inv_cov @ ones / (ones @ inv_cov @ ones)
w_mkv = pd.Series(w_mkv_values, index=codes)

weights = pd.DataFrame({
    '等权': w_equal, '反向波动': w_inv_vol, 'Markowitz': w_mkv
})
print("三种权重方案对比:")
print(weights.round(3))
print(f"\n权重总和（应 = 1.0）:")
print(weights.sum().round(6))

## 第 4 段：组合回测 + 再平衡

假设每月（22 交易日）再平衡一次。

In [ ]:
def backtest_portfolio(rets: pd.DataFrame, weights: pd.Series,
                      rebalance_freq: int = 22) -> pd.Series:
    \"\"\"组合回测：定期再平衡到目标权重。\"\"\"
    nav = 1.0
    navs = []
    w = weights.copy()
    for i, (date, daily_rets) in enumerate(rets.iterrows()):
        # 当日组合收益
        port_ret = (w * daily_rets).sum()
        nav *= (1 + port_ret)
        navs.append(nav)
        # 权重随涨跌漂移
        w = w * (1 + daily_rets)
        w = w / w.sum()
        # 到再平衡日，重置回目标权重
        if (i + 1) % rebalance_freq == 0:
            w = weights.copy()
    return pd.Series(navs, index=rets.index, name='nav')

nav_equal = backtest_portfolio(rets, w_equal)
nav_inv_vol = backtest_portfolio(rets, w_inv_vol)
nav_mkv = backtest_portfolio(rets, w_mkv)
nav_bench = (1 + rets.mean(axis=1)).cumprod()  # 等权基准作为对照

fig, ax = plt.subplots(figsize=(13, 6))
nav_equal.plot(ax=ax, label='等权', linewidth=2)
nav_inv_vol.plot(ax=ax, label='反向波动', linestyle='--')
nav_mkv.plot(ax=ax, label='Markowitz 最小方差', linestyle='-.')
ax.set_title('三种组合构建方案净值对比')
ax.legend()
plt.tight_layout(); plt.show()

## 第 5 段：组合绩效评估

用 qtrader.metrics 的扩展指标评估三种方案。

In [ ]:
from qtrader.metrics import compute_metrics

results = {}
for name, nav in [('等权', nav_equal), ('反向波动', nav_inv_vol), ('Markowitz', nav_mkv)]:
    strat_ret = nav.pct_change().fillna(0)
    # 传 nav 与 bench_ret（这里用等权作为基准对照）
    m = compute_metrics(strat_ret, nav, nav_equal.pct_change().fillna(0))
    results[name] = m

import pandas as pd
summary = pd.DataFrame({
    name: {k: v for k, v in m.items() if k in
           ['total_return', 'ann_return', 'sharpe', 'sortino', 'calmar',
            'max_drawdown', 'volatility', 'win_rate']}
    for name, m in results.items()
}).T
print("三种组合方案绩效对比:")
print(summary.round(3))

## 第 6 段：本节要点 + Phase 2 收官

### 组合构建要点
1. **等权 > 你以为**：简单方案常常打败精巧的 Markowitz
2. **反向波动是稳健中间路线**：考虑风险差异但不过度优化
3. **Markowitz 对噪声敏感**：必须 shrinkage 协方差矩阵或用 Black-Litterman
4. **再平衡是 alpha 来源**：定期「低买高卖」本质 = 逆向投资

### Phase 2 收官
恭喜走完 L11-L15！现在你拥有了完整的量化基础栈：
- **数据层**：akshare / 多股对齐 / 复权（Phase 1）
- **指标层**：风险/收益/成本三件套（Phase 1 L06 + Phase 2 L11 metrics 扩展）
- **策略层**：择时三巨头（DualMA/Turtle/Grid）+ 选股 + 组合（Phase 2）
- **回测层**：向量化引擎 + CostModel + Walk-Forward（qtrader 框架）

### Phase 3 预告（未来阶段）
- 机器学习 Alpha 因子（LightGBM、强化学习）
- 事件驱动引擎（限价单、涨跌停拒单）
- 实盘对接（券商 API、CTP）
- 衍生品（可转债、股指期货）

### 📝 `exercises/ex14.py` / `ex15.py`
本阶段配套 2 份课后练习，请独立完成。